In [1]:
import pandas as pd
import re
from pathlib import Path


In [ ]:
# ===================== CONFIG PARAM ====================
DS_NAME = "Apache"
TEMP_PATH = Path(f"../data/{DS_NAME}/{DS_NAME}_full.log_templates.csv")
STRU_PATH = Path(f"../data/{DS_NAME}/{DS_NAME}_full.log_structured.csv")
RAWL_PATH  = Path(f"../data/{DS_NAME}/{DS_NAME}_full.log")



OUT_DIR = Path("../data_prc/" + DS_NAME)
OUT_DIR.mkdir(parents=True, exist_ok=True)
# =======================================================

In [3]:
def tpl2rex(tpl):
    tpl_rex = re.sub(r"<.{1,5}>", "<*>", tpl)
    tpl_rex = re.sub(r"([^A-Za-z0-9])", r"\\\1", tpl_rex)
    tpl_rex = re.sub(r"\\ +",           r"\\s+", tpl_rex)
    tpl_rex = ("^" +tpl_rex.replace(r"\<\*\>", "(.*?)") + "$")

    return re.compile(tpl_rex)

def createNumUniCol(temp_df=None, stru_df=None):
    if temp_df is None or stru_df is None:
        return None
    else:
        unique_counts = (
            stru_df.groupby("EventId")["Content"]
                .nunique()
                .to_dict()
        )
        temp_df["NumUniques"] = temp_df["EventId"].map(unique_counts).fillna(0).astype(int)

        temp_df["NumofParams"] = (
            temp_df["EventTemplate"]
                .astype(str)
                .str.count(r"<\*>")
        )
def mainPrc():
    temp_df = pd.read_csv(TEMP_PATH)
    stru_df = pd.read_csv(STRU_PATH)
    
    createNumUniCol(temp_df, stru_df)
    
    for idx, tpl_row in temp_df.iterrows():
        eid   = tpl_row["EventId"]
        e_tpl = tpl_row["EventTemplate"]
        occur = tpl_row["Occurrences"]
        
        regex = tpl2rex(e_tpl)
        
        logs  = stru_df.loc[stru_df["EventId"] == eid, "Content"]
        uni_logs  = logs.unique().tolist()
        temp_df.loc[idx, "NumUniques"] = len(uni_logs)
        
        out_file = Path(OUT_DIR / f"{eid}.txt")
        with open(out_file, "w", encoding="utf-8") as f:
            f.write(f"EventId: {eid}\n")
            f.write(f"EventTemplate: {e_tpl}\n")
            f.write(f"Occurrences: {occur}\n\n")
            f.write(f"Nums of unique: {len(uni_logs)}\n")
            
            f.write("Example sample logs (<30): \n")

            for i, log in enumerate(uni_logs):
                m = regex.match(log)
                if m:
                    params = list(m.groups())
                else: 
                    params = []
                    
                line = log + "\t"
                for p in params:
                    line += f"|{p}"
                f.write(line + "\n")
                
                if i > 30: 
                    break
    out_csv = TEMP_PATH.with_name(
        TEMP_PATH.stem + "_stats.csv"
    )

    temp_df.to_csv(out_csv, index=False)

In [4]:
mainPrc()

In [ ]:
tpl_df = Path("../data/Apache/Apache_full.log_templates_stats.csv")
tpl_df = pd.read_csv(tpl_df)

tpl_df.head(30)

,EventId,EventTemplate,Occurrences,NumUniques,NumofParams
0,E20,LDAP: Built with OpenLDAP LDAP SDK,53,1,0
1,E23,LDAP: SSL support unavailable,53,1,0
2,E18,suEXEC mechanism enabled (wrapper: <*>),8,1,1
3,E4,Digest: generating secret for digest authentic...,45,1,0
4,E29,Digest: done,45,1,0
5,E10,env.createBean2(): Factory error creating <*> ...,180,4,3
6,E21,config.update(): Can't create <*>,180,4,1
7,E2,mod_python: Creating <*> session mutexes based...,45,1,3
8,E25,mod_security/<*> configured,45,1,1
9,E8,Apache/<*> configured -- resuming normal opera...,45,1,1
